# Duka Electronics: H1 2026 Sales Exploration

In [18]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1000
products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500,
            "Tablet": 28000, "Charger": 1200}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n,
                         p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n,
                          p=[0.55, 0.35, 0.10]),
})
sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

## Executive Summary
- Duka Electronics sold KES 71.8M across 1,000 orders in January–June 2026.
- Nairobi is the leading branch with KES 26.1M (36.4% of revenue), and the branch ranking is identical
  whether measured by revenue or by number of orders.
- Laptops and Tablets generate 84.8% of revenue (KES 60.9M) from only 44% of units sold, so the
  accessories (Headphones, Chargers) matter little for revenue.
- April was the best month at KES 13.52M, 24.0% above June (KES 10.90M), though June has only 29 days of data.
- Premium online orders from Nairobi and Mombasa total 83 orders and KES 10.2M, which is 14.2% of all revenue.

## 1. First-five-minutes inspection.

In [19]:
import numpy as np
import pandas as pd

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("shape:", df.shape)
display(df.head())
display(df.dtypes)
print("missing values:\n", df.isna().sum())
print("duplicate order_id:", df["order_id"].duplicated().sum())
print("date range:", df["order_date"].min(), "->", df["order_date"].max())
for col in ["branch", "product", "channel"]:
    display(df[col].value_counts())
display(df[["quantity", "unit_price"]].describe())

shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,0
order_id,int64
order_date,datetime64[ns]
branch,object
product,object
quantity,int64
channel,object
unit_price,int64


missing values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64
duplicate order_id: 0
date range: 2026-01-01 00:00:00 -> 2026-06-29 00:00:00


,count
branch,
Nairobi,376
Mombasa,186
Nakuru,162
Kisumu,159
Eldoret,117


,count
product,
Laptop,220
Tablet,215
Phone,194
Headphones,187
Charger,184


,count
channel,
In-store,579
Online,320
Phone order,101


,quantity,unit_price
count,1000.000000,1000.000000
mean,3.012000,23918.300000
std,1.416991,23908.790546
min,1.000000,1200.000000
25%,2.000000,2500.000000
50%,3.000000,15000.000000
75%,4.000000,28000.000000
max,5.000000,65000.000000


## 2. Derived columns.

In [20]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.strftime("%Y-%m")
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

display(df.head())
print(df.groupby("product")["price_band"].first())
print("TOTAL REVENUE:", df["revenue"].sum())

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,2026-03,Saturday,True,Value


product
Charger         Value
Headphones      Value
Laptop        Premium
Phone           Value
Tablet        Premium
Name: price_band, dtype: object
TOTAL REVENUE: 71792500


## 3. Branch ranking: revenue vs number of orders

In [21]:
branch = (
    df.groupby("branch")
      .agg(total_revenue=("revenue", "sum"), n_orders=("order_id", "count"))
)
branch["avg_order_value"] = branch["total_revenue"] / branch["n_orders"]
branch["revenue_share_%"] = branch["total_revenue"] / branch["total_revenue"].sum() * 100
branch["orders_share_%"] = branch["n_orders"] / branch["n_orders"].sum() * 100
branch["revenue_rank"] = branch["total_revenue"].rank(ascending=False, method="min").astype(int)
branch["orders_rank"] = branch["n_orders"].rank(ascending=False, method="min").astype(int)

branch = branch.sort_values("total_revenue", ascending=False).round(1)
display(branch)
print("same ranking?", (branch["revenue_rank"] == branch["orders_rank"]).all())

,total_revenue,n_orders,avg_order_value,revenue_share_%,orders_share_%,revenue_rank,orders_rank
branch,,,,,,,
Nairobi,26106100,376,69431.1,36.4,37.6,1,1
Mombasa,14003700,186,75288.7,19.5,18.6,2,2
Nakuru,11970400,162,73891.4,16.7,16.2,3,3
Kisumu,10868100,159,68352.8,15.1,15.9,4,4
Eldoret,8844200,117,75591.5,12.3,11.7,5,5


same ranking? True


 Yes, the two rankings are identical: Nairobi, Mombasa, Nakuru, Kisumu, Eldoret.
Nairobi leads with KES 26.1M (36.4% of revenue) from 376 orders (37.6% of orders), almost double Mombasa's
KES 14.0M. The rankings match because order volume differs far more between branches than order value does:
average order value only ranges from KES 68,353 (Kisumu) to 75,592 (Eldoret), about ±5% around the
KES 71,793 overall average, which is too narrow to reorder branches whose order counts differ by 20–100%.
Revenue shares do shift slightly: Nairobi's revenue share (36.4%) is below its order share (37.6%) because
its average order is below the average, while Mombasa and Eldoret earn more than their order share.
Nakuru and Kisumu are within 3 orders of each other, so Nakuru's lead (KES 1.1M) comes mostly from its
larger average order. Eldoret has the highest average order but the fewest orders, so higher basket value
did not outweigh volume.

# 4. Top products: revenue vs units sold

In [22]:
product = (
    df.groupby("product")
      .agg(total_revenue=("revenue", "sum"),
           units_sold=("quantity", "sum"),
           n_orders=("order_id", "count"))
)
product["revenue_share_%"] = (product["total_revenue"] / product["total_revenue"].sum() * 100).round(1)

top_revenue = product.sort_values("total_revenue", ascending=False).head(3)
top_units = product.sort_values("units_sold", ascending=False).head(3)

display(top_revenue)
display(top_units)

,total_revenue,units_sold,n_orders,revenue_share_%
product,,,,
Laptop,41535000,639,220,57.9
Tablet,19348000,691,215,26.9
Phone,8880000,592,194,12.4


,total_revenue,units_sold,n_orders,revenue_share_%
product,,,,
Tablet,19348000,691,215,26.9
Laptop,41535000,639,220,57.9
Phone,8880000,592,194,12.4


By revenue the top three are Laptop (KES 41.5M, 57.9% of revenue), Tablet (KES 19.3M, 26.9%)
and Phone (KES 8.9M, 12.4%). By units sold the top three are Tablet (691), Laptop (639) and Phone (592).
The same three products appear in both lists, but Laptop and Tablet swap first and second place.
The difference is price: a Laptop costs KES 65,000, 2.3 times the Tablet's KES 28,000, so Laptop's 639 units
earn more than twice what Tablet's 691 units do, despite 52 fewer units sold. Headphones and Charger
are cheap (KES 2,500 and 1,200) and make up about a third of all units sold but only about 3% of revenue.
For the Head of Sales this means the unit ranking is misleading: Laptops and Tablets together
bring in about 85% of revenue, so stock, promotion and availability of these two matter far more than
how many accessories are sold. The unit counts are close (592 to 691 across the top three), so the units
ordering is probably close to random; the revenue ordering is driven by price and is reliable.

## 5. Weekend vs weekday orders

In [23]:
wk = (
    df.groupby("is_weekend")
      .agg(n_orders=("order_id", "count"),
           total_revenue=("revenue", "sum"),
           avg_order_revenue=("revenue", "mean"))
      .rename(index={False: "Weekday", True: "Weekend"})
)
wk["order_share_%"] = wk["n_orders"] / wk["n_orders"].sum() * 100
wk["revenue_share_%"] = wk["total_revenue"] / wk["total_revenue"].sum() * 100

# fair comparison: weekends are only 2 of every 7 days
all_days = pd.date_range(df["order_date"].min(), df["order_date"].max())
n_weekend_days = int((all_days.dayofweek >= 5).sum())
wk["calendar_days"] = pd.Series({"Weekday": len(all_days) - n_weekend_days,
                                 "Weekend": n_weekend_days})
wk["orders_per_day"] = wk["n_orders"] / wk["calendar_days"]
wk["revenue_per_day"] = wk["total_revenue"] / wk["calendar_days"]

display(wk.round(1))

,n_orders,total_revenue,avg_order_revenue,order_share_%,revenue_share_%,calendar_days,orders_per_day,revenue_per_day
is_weekend,,,,,,,,
Weekday,723,51856200,71723.7,72.3,72.2,128,5.6,405126.6
Weekend,277,19936300,71972.2,27.7,27.8,52,5.3,383390.4


 Weekdays had 723 orders and KES 51.9M in revenue (average order KES 71,724); weekends had 277 orders
and KES 19.9M (average order KES 71,972). Weekends account for 27.7% of orders and 27.8% of revenue, which is
close to their 28.9% share of calendar days (52 of 180). Adjusted for the number of days, weekends are
slightly slower, with 5.3 orders per day against 5.6 on weekdays and about 5% less revenue per day.
Average order value is almost identical (a 0.3% difference), so weekend customers are not buying bigger
or smaller baskets. The weekend shortfall of about 12 orders compared with an even spread is small and within
normal random variation, so the data gives no evidence of a real weekend slump. Staffing and promotions
should not be shifted on this basis.

## 6. Best and worst months

In [24]:
monthly = df.groupby("month").agg(total_revenue=("revenue", "sum"),
                                  n_orders=("order_id", "count"))
monthly["days_covered"] = pd.Series(all_days.strftime("%Y-%m")).value_counts().sort_index()
monthly["revenue_per_day"] = (monthly["total_revenue"] / monthly["days_covered"]).round(0)
display(monthly)

best, worst = monthly["total_revenue"].idxmax(), monthly["total_revenue"].idxmin()
best_rev, worst_rev = monthly.loc[best, "total_revenue"], monthly.loc[worst, "total_revenue"]
print("best month: ", best, best_rev)
print("worst month:", worst, worst_rev)
print("best is higher than worst by:", round((best_rev - worst_rev) / worst_rev * 100, 1), "%")
print("worst is lower than best by: ", round((best_rev - worst_rev) / best_rev * 100, 1), "%")

,total_revenue,n_orders,days_covered,revenue_per_day
month,,,,
2026-01,11786600,173,31,380213.0
2026-02,11772000,159,28,420429.0
2026-03,12396800,186,31,399897.0
2026-04,13518300,167,30,450610.0
2026-05,11415400,153,31,368239.0
2026-06,10903400,162,29,375979.0


best month:  2026-04 13518300
worst month: 2026-06 10903400
best is higher than worst by: 24.0 %
worst is lower than best by:  19.3 %


 The best month for revenue is April 2026 at KES 13.52M and the worst is June at KES 10.90M.
April is 24.0% higher than June (equivalently, June is 19.3% lower than April; I use the worst month as the
base for the first figure). Month lengths matter: June has only 29 days of data and February has 28,
so I also checked revenue per day. April is still best (KES 450,610/day), but May is the lowest
(KES 368,239/day), about 22% below April, while June (KES 375,979/day) is slightly above it. April's lead comes from
order value, not volume: it had 167 orders, fewer than March (186), but a richer mix of laptop and tablet
orders. With only six months and 150-190 orders in each, gaps of this size can happen by chance,
so I would not read this as a seasonal pattern without more history.

## 7. Premium online orders, Nairobi and Mombasa

In [25]:
import os
os.makedirs("outputs", exist_ok=True)

mask = (
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
)
premium_online = df[mask]

print("orders:", len(premium_online))
print("total revenue:", premium_online["revenue"].sum())
print("share of all revenue (%):", round(premium_online["revenue"].sum() / df["revenue"].sum() * 100, 1))
display(premium_online.groupby("branch").agg(n_orders=("order_id", "count"),
                                              revenue=("revenue", "sum")))

premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)

orders: 83
total revenue: 10218000
share of all revenue (%): 14.2


,n_orders,revenue
branch,,
Mombasa,33,3696000
Nairobi,50,6522000


In [26]:
check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print(check.shape)
print(check["price_band"].unique(), check["channel"].unique(), check["branch"].unique())


(83, 12)
['Premium'] ['Online'] ['Nairobi' 'Mombasa']


There are 83 Premium online orders from Nairobi and Mombasa, worth KES 10,218,000 in total,
14.2% of all Duka revenue. Nairobi accounts for 50 of these orders (KES 6.52M, about 64% of the group's revenue)
and Mombasa for 33 (KES 3.70M, about 36%). Nairobi's larger share follows its larger overall order volume.
Nairobi's average order here (about KES 130,000) is also higher than Mombasa's (about KES 112,000).
The filtered orders are saved to outputs/premium_online_coast_capital.csv.

In [27]:
cols = ["order_id", "order_date", "branch", "product", "quantity", "channel", "unit_price"]
df[cols].to_excel("duka_data_for_excel.xlsx", sheet_name="Data", index=False)

In [28]:
import os
os.makedirs("excel", exist_ok=True)

## 8. Spreadsheet check: Excel vs pandas

In [29]:
xl_path = "excel/duka_branch_summary.xlsx"

xl_branch = pd.read_excel(xl_path, sheet_name="Branch Summary", usecols="A:C", nrows=6)
xl_branch.columns = ["branch", "excel_revenue", "excel_region"]
xl_branch = xl_branch[xl_branch["branch"] != "Total"].set_index("branch")

cmp = xl_branch.join(df.groupby("branch")["revenue"].sum().rename("pandas_revenue"))
cmp["match"] = cmp["excel_revenue"] == cmp["pandas_revenue"]
display(cmp)

print("Excel total: ", xl_branch["excel_revenue"].sum())
print("pandas total:", df["revenue"].sum())
print("All branch totals match:", cmp["match"].all())

,excel_revenue,excel_region,pandas_revenue,match
branch,,,,
Nairobi,26106100,Nairobi,26106100,True
Mombasa,14003700,Coast,14003700,True
Nakuru,11970400,Rift Valley,11970400,True
Kisumu,10868100,Western,10868100,True
Eldoret,8844200,Rift Valley,8844200,True


Excel total:  71792500
pandas total: 71792500
All branch totals match: True


In [30]:
raw = pd.read_excel(xl_path, sheet_name="Pivot", header=None)
display(raw)

excel_gt = raw[raw[0] == "Grand Total"].iloc[0, 1:].dropna().astype(float).tolist()

by_channel = df.groupby("channel")["revenue"].sum().sort_index()
pandas_gt = by_channel.tolist() + [df["revenue"].sum()]

print("Excel pivot channel + grand totals: ", excel_gt)
print("pandas channel + grand totals:      ", pandas_gt)
print("Pivot totals match pandas:", excel_gt == pandas_gt)

,0,1,2,3,4
0,SUM of Revenue,channel,NaN,NaN,NaN
1,branch,In-store,Online,Phone order,Grand Total
2,Eldoret,4885300,3450700,508200,8844200
3,Kisumu,6468600,3370800,1028700,10868100
4,Mombasa,8039600,4086800,1877300,14003700
5,Nairobi,14808400,7527400,3770300,26106100
6,Nakuru,6767300,4350400,852700,11970400
7,Grand Total,40969200,22786100,8037200,71792500


Excel pivot channel + grand totals:  [40969200.0, 22786100.0, 8037200.0, 71792500.0]
pandas channel + grand totals:       [40969200, 22786100, 8037200, np.int64(71792500)]
Pivot totals match pandas: True


In [31]:
top2 = product.loc[["Laptop", "Tablet"]]
print(top2["total_revenue"].sum(), top2["total_revenue"].sum() / df["revenue"].sum() * 100)
print(top2["units_sold"].sum() / df["quantity"].sum() * 100)

60883000 84.80412299334888
44.156706507304115
